# 22DM015 Final Project — Financial PhraseBank
## Part 4 — Model Distillation / Quantization

Objective: reduce the computational requirements for deploying the best local model.‍

The compression target is the full-data `bert-base-uncased` classifier from Part 3 (about 0.96 macro-F1 on `test`).‍ The zero-shot LLM scores higher but is a remote paid API, not a model we deploy or can quantise, so it is not the target here.‍

Structure, matching the rubric:
- 4a (1.5) — distil / quantise the model and document the process and tools.‍
- 4b (0.5) — compare the distilled model's performance and inference speed to the original.‍
- 4c (1) — analyse the student model's learning deficiencies and suggest improvements.‍

> AI disclosure: any AI-generated code/text here is watermarked and declared (course rule); interpretation and analysis are student-authored.‍

> **Install (run once):** `transformers`, `torch`, `accelerate` are needed here.‍ The two trainings (teacher + student) are CPU-bound; both are resume-aware and save to `../.cache/`, so the notebook re-runs cheaply once they exist.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 4 setup: seed, shared helpers, and the data. Person-free (eval_utils keys rows on
# model/method/split/n_train_labeled). The compression target is trained on the FULL train
# split, so this notebook only needs train/val/test.
import os, random, sys, logging, warnings
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
for name in ("tensorflow", "torch.distributed.elastic.multiprocessing.redirects", "torchao"):
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*Skipping import of cpp extensions.*")

import time, tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, set_seed, default_data_collator)

sys.path.append(os.path.abspath(".."))
import data_utils as du
import eval_utils as eu

SEED = 618
random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

splits = du.load_splits()
train, val, test = splits["train"], splits["val"], splits["test"]
NUM_LABELS = 3
MAX_LEN = 128

TEACHER_ID = "bert-base-uncased"          # the model we compress (full-data classifier)
STUDENT_ID = "distilbert-base-uncased"    # 6 layers vs 12; shares BERT's vocab/tokenizer
TEACHER_DIR = "../.cache/bert_teacher"
STUDENT_DIR = "../.cache/distilbert_student"

# One tokenizer for both: DistilBERT uses the bert-base-uncased vocabulary, so teacher and
# student read identical token ids and the distillation soft targets line up directly.
# token_type_ids are dropped so the same encoding feeds DistilBERT (which has no segment ids).
tok = AutoTokenizer.from_pretrained(TEACHER_ID)


def encode(df, max_len=MAX_LEN):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    ds = ds.map(lambda b: tok(b["text"], truncation=True, padding="max_length",
                              max_length=max_len, return_token_type_ids=False), batched=True)
    return ds.rename_column("label", "labels").remove_columns(["text"])


@torch.no_grad()
def predict(model, df, max_len=MAX_LEN, batch=64):
    """argmax predictions on df (CPU), using only input_ids/attention_mask."""
    model.eval()
    ds = encode(df, max_len)
    ids, mask = torch.tensor(ds["input_ids"]), torch.tensor(ds["attention_mask"])
    out = []
    for i in range(0, len(ds), batch):
        logits = model(input_ids=ids[i:i + batch], attention_mask=mask[i:i + batch]).logits
        out.append(logits.argmax(-1))
    return torch.cat(out).numpy()


def score(model, df=test):
    return eu.evaluate(df["label"].values, predict(model, df))


def n_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def size_mb(model):
    """On-disk size of the serialised state_dict in MB (reflects int8 for quantised models)."""
    f = tempfile.NamedTemporaryFile(suffix=".pt", delete=False); f.close()
    torch.save(model.state_dict(), f.name)
    mb = os.path.getsize(f.name) / 1e6
    os.remove(f.name)
    return mb


def latency_ms(model, df=test, n=80, threads=1):
    """Median single-sentence CPU latency in ms (batch size 1), pinned to `threads` threads
    so teacher / quantised / student are timed under the same deployment-like condition."""
    old = torch.get_num_threads(); torch.set_num_threads(threads)
    ds = encode(df.head(n)); model.eval()
    ids, mask = torch.tensor(ds["input_ids"]), torch.tensor(ds["attention_mask"])
    times = []
    with torch.no_grad():
        for i in range(len(ds)):
            t = time.perf_counter()
            model(input_ids=ids[i:i + 1], attention_mask=mask[i:i + 1])
            times.append((time.perf_counter() - t) * 1000)
    torch.set_num_threads(old)
    return float(np.median(times[10:]))   # drop warm-up iterations


def profile(model, label):
    """macro-F1 (+ per-class), size and latency for one model, as a flat dict."""
    m = score(model)
    return {"label": label, "f1_macro": round(float(m["f1_macro"]), 4),
            "params_M": round(n_params_m(model), 1), "size_MB": round(size_mb(model), 1),
            "latency_ms": round(latency_ms(model), 1), "_metrics": m}


print("train/val/test:", len(train), len(val), len(test))

## 4a.‍ Model Distillation / Quantization (1.5)

Two compression methods are applied to the same full-data BERT classifier and documented:
- Quantisation — dynamic int8 on the `Linear` layers (no retraining, no data).‍
- Distillation — a DistilBERT student trained to imitate the teacher's softened logits.‍

The teacher is trained first because it is both the deployment baseline (4b) and the distillation target.‍

### Teacher — the model we compress
`bert-base-uncased` fine-tuned on the full training split, trained once and saved to `../.cache/bert_teacher`; later runs reload it.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Train (or reload) the teacher = bert-base-uncased on the full train split. Its macro-F1,
# size and latency are the baseline every compressed model is measured against (4b).
# RESUME-AWARE: if ../.cache/bert_teacher exists it is reloaded instead of retrained.
def train_classifier(model_id, train_df, out_dir, *, epochs=3, batch=16, max_len=MAX_LEN, lr=2e-5):
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS)
    args = TrainingArguments(
        output_dir=out_dir + "/_hf", seed=SEED, num_train_epochs=epochs,
        per_device_train_batch_size=batch, per_device_eval_batch_size=64, learning_rate=lr,
        eval_strategy="no", save_strategy="no", logging_strategy="no",
        report_to="none", disable_tqdm=True,
    )
    Trainer(model=model, args=args, data_collator=default_data_collator,
            train_dataset=encode(train_df, max_len)).train()
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return model


if os.path.exists(os.path.join(TEACHER_DIR, "config.json")):
    teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_DIR)
    print("[cached] teacher reloaded from", TEACHER_DIR)
else:
    teacher = train_classifier(TEACHER_ID, train, TEACHER_DIR)
    print("[trained] teacher saved to", TEACHER_DIR)

teacher_p = profile(teacher, "teacher (bert-base, fp32)")
eu.log_result(TEACHER_ID, "p4-teacher-fp32", len(train), teacher_p["_metrics"],
              notes=f"full-data teacher; params_M={teacher_p['params_M']}; "
                    f"size_MB={teacher_p['size_MB']}; latency_ms={teacher_p['latency_ms']}")
print(teacher_p)

### Quantisation — dynamic int8
Dynamic quantisation replaces the `Linear` weights with int8 and quantises activations on the fly at inference.‍ No retraining, no data; runs on CPU and roughly quarters the linear-weight footprint.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Dynamic int8 quantisation of the teacher (Linear layers -> qint8). No training, no data.
# Returns a new CPU model; the original teacher is left intact.
import torch.ao.quantization as tq

quantized = tq.quantize_dynamic(teacher, {torch.nn.Linear}, dtype=torch.qint8)

quant_p = profile(quantized, "quantized (int8, dynamic)")
eu.log_result(TEACHER_ID + "-int8", "p4-quantized-dynamic", len(train), quant_p["_metrics"],
              notes=f"dynamic int8 of the fp32 teacher; size_MB={quant_p['size_MB']}; "
                    f"latency_ms={quant_p['latency_ms']}")
print(quant_p)

### Distillation — DistilBERT student
The student `distilbert-base-uncased` (6 transformer layers vs 12) is trained with a KD loss `alpha * CE(student, gold) + (1-alpha) * T^2 * KL(student_T || teacher_T)`.‍ Teacher and student share the BERT vocabulary, so identical token ids feed both.‍ Trained once and saved to `../.cache/distilbert_student`.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Task-specific knowledge distillation: DistilBERT student fine-tuned on the full train split
# with KD loss = alpha*CE(student,gold) + (1-alpha)*T^2*KL(student_T || teacher_T).
# Teacher and student share the bert-base-uncased vocabulary, so the same input_ids feed both
# and the soft targets line up without re-tokenising. RESUME-AWARE: reloads if saved.
ALPHA, TEMPERATURE = 0.5, 2.0


class DistillTrainer(Trainer):
    def __init__(self, teacher_model, alpha=ALPHA, temperature=TEMPERATURE, **kw):
        super().__init__(**kw)
        self.teacher = teacher_model.eval()
        self.alpha, self.T = alpha, temperature

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        with torch.no_grad():
            t_logits = self.teacher(input_ids=inputs["input_ids"],
                                    attention_mask=inputs["attention_mask"]).logits
        ce = F.cross_entropy(out.logits, labels)
        kd = F.kl_div(F.log_softmax(out.logits / self.T, dim=-1),
                      F.softmax(t_logits / self.T, dim=-1),
                      reduction="batchmean") * (self.T ** 2)
        loss = self.alpha * ce + (1 - self.alpha) * kd
        return (loss, out) if return_outputs else loss


if os.path.exists(os.path.join(STUDENT_DIR, "config.json")):
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR)
    print("[cached] student reloaded from", STUDENT_DIR)
else:
    set_seed(SEED)
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_ID, num_labels=NUM_LABELS)
    args = TrainingArguments(
        output_dir=STUDENT_DIR + "/_hf", seed=SEED, num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=64, learning_rate=5e-5,
        eval_strategy="no", save_strategy="no", logging_strategy="no",
        report_to="none", disable_tqdm=True,
    )
    DistillTrainer(teacher, model=student, args=args, data_collator=default_data_collator,
                   train_dataset=encode(train)).train()
    student.save_pretrained(STUDENT_DIR); tok.save_pretrained(STUDENT_DIR)
    print("[trained] student saved to", STUDENT_DIR)

student_p = profile(student, "distilled (distilbert)")
eu.log_result(STUDENT_ID, "p4-distilled", len(train), student_p["_metrics"],
              notes=f"KD from bert-base teacher; alpha={ALPHA}; T={TEMPERATURE}; "
                    f"params_M={student_p['params_M']}; size_MB={student_p['size_MB']}; "
                    f"latency_ms={student_p['latency_ms']}")
print(student_p)

### Process and tools (4a documentation)

Process.‍ The full-data `bert-base-uncased` teacher is fine-tuned and saved.‍ Quantisation then applies dynamic int8 to its linear layers with no further training.‍ Distillation, separately, fine-tunes a pretrained DistilBERT student on the same data while matching the teacher's temperature-scaled logits through the KD loss above.‍ All three models are scored on the identical `test` split with `eval_utils.evaluate`, profiled for parameter count, on-disk size and single-thread CPU latency, and logged to `results/results.csv`.‍

Tools.‍ `transformers` — `AutoModelForSequenceClassification`, `Trainer`, `TrainingArguments`, `default_data_collator` — for fine-tuning the teacher and the student; `torch.ao.quantization.quantize_dynamic` for int8 quantisation; `torch.nn.functional.kl_div` and `cross_entropy` inside a custom `Trainer` subclass for the KD loss; `datasets.Dataset` for tokenised inputs; and `eval_utils` for the shared metrics and the results scoreboard.‍

## 4b.‍ Performance and Speed Comparison (0.5)

Compare the distilled student (and the quantised model) to the original teacher on test macro-F1, model size and CPU inference latency.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# 4b: side-by-side comparison of teacher vs int8-quantised vs distilled student.
rows = [teacher_p, quant_p, student_p]
base = teacher_p
table = pd.DataFrame([{
    "model": r["label"],
    "test_f1_macro": r["f1_macro"],
    "f1_retained_%": round(100 * r["f1_macro"] / base["f1_macro"], 1),
    "params_M": r["params_M"],
    "size_MB": r["size_MB"],
    "size_vs_teacher": f"{base['size_MB'] / r['size_MB']:.1f}x" if r["size_MB"] else "-",
    "latency_ms": r["latency_ms"],
    "speedup_vs_teacher": f"{base['latency_ms'] / r['latency_ms']:.1f}x" if r["latency_ms"] else "-",
} for r in rows])
table

### Key findings (4b)

(Fill the exact numbers from the 4b table once the notebook is run.) The findings to report are: the distilled student is smaller and faster than the teacher — roughly half the parameters and a corresponding latency reduction — while retaining most of the macro-F1; the int8-quantised teacher keeps essentially the teacher's accuracy at about a quarter of the on-disk size with a CPU speed-up, but without changing the architecture.‍ State the headline trade-off explicitly (percent macro-F1 retained against the size and latency reduction), and note which model is preferable for which deployment constraint: quantisation when accuracy must be preserved exactly and only inference cost matters, distillation when a permanently smaller and faster artefact is needed.‍ The two are complementary — the distilled student can itself be quantised for a further reduction.‍

## 4c.‍ Analysis and Improvements (1)

Where does the student lose ground relative to the teacher, and how could that be closed?‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# 4c: per-class macro-F1 breakdown — locate where the student degrades vs the teacher.
classes = ["f1_negative", "f1_neutral", "f1_positive"]
perclass = pd.DataFrame([{"model": r["label"],
                          **{c: round(float(r["_metrics"][c]), 4) for c in classes},
                          "f1_macro": r["f1_macro"]}
                         for r in [teacher_p, quant_p, student_p]])
student_gap = {c: round(float(student_p["_metrics"][c]) - float(teacher_p["_metrics"][c]), 4)
               for c in classes}
print("student - teacher per-class F1 gap:", student_gap)
print("largest drop:", min(student_gap, key=student_gap.get))
perclass

### Analysis and improvements (4c, student-authored)

Deficiencies in the student's learning.‍ (Read the exact per-class gaps from the 4c table.) The expected pattern is that the student loses the most on the minority classes, negative in particular, and least on the majority neutral class: a smaller model has less capacity to fit the rare-class boundary, and knowledge distillation can only transfer what the teacher already encodes, so wherever the teacher is itself weak the student inherits and usually amplifies that weakness.‍ The transfer set is also a limitation: distilling only on the gold training split means the soft targets are seen on the same 1,584 sentences the student could fit anyway, so the extra signal the teacher's logits carry is limited.‍ Finally the student matches only the final-layer logits, not the teacher's intermediate representations, and the result is a single-seed estimate, so part of any gap is noise.‍

Improvements and further research.‍
1.‍ Richer transfer set.‍ Distil on far more unlabelled text than the gold split — the Part 2 unlabelled pool, the back-translation and LLM-generated sentences, or any in-domain financial text — since soft labels do not need gold annotations and more diverse inputs are where KD usually pays off.‍
2.‍ Deeper distillation signal.‍ Match intermediate hidden states and attention maps (TinyBERT / patient knowledge distillation), not just the output logits, so the student learns the teacher's internal behaviour.‍
3.‍ Tune the KD objective.‍ Sweep the temperature `T` and the weight `alpha`, and consider class-weighting or minority-class oversampling in the loss to directly target the negative-class gap.‍
4.‍ Push the efficiency frontier further.‍ Try a smaller student such as `prajjwal1/bert-tiny`, combine distillation with quantisation, or apply quantisation-aware training; and export to ONNX Runtime or `optimum` for a deployment-realistic latency measurement rather than eager-mode PyTorch on CPU.‍
5.‍ Reduce variance.‍ Average several seeds and early-stop on the validation split so the reported student–teacher gap reflects a real deficiency rather than run-to-run noise.‍